<a href="https://colab.research.google.com/github/fsemecurbe/AOTMS/blob/main/Download_OCSGE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [49]:
#pip install py7zr
import geopandas as gpd
import pandas as pd
import requests
from bs4 import BeautifulSoup
import zipfile
import py7zr
import os

In [4]:
req = "https://geoservices.ign.fr/ocsge"

In [5]:
page = requests.get(req)

In [6]:
soup = BeautifulSoup(page.text, 'html.parser')

In [7]:
downloads = soup.findAll('a', href=True)

Liste des départements disponibles

In [8]:
liste_ocsge = [download for download in downloads if "OCS-GE_2-0" in download['href']]
liste_ocsge = pd.DataFrame({ 'name' : [item.find('strong').text if item.find('strong') else None for item in liste_ocsge],
               'link' : [item['href'] for item in liste_ocsge] })

temp = liste_ocsge.name.str.split('-', expand=True)
liste_ocsge['codgeo'] = temp[0].str.slice(12,).str.strip()
liste_ocsge['libgeo'] = temp[1].str.strip()
liste_ocsge['year'] =  temp[2].str.strip()
liste_ocsge.loc[liste_ocsge.year=="différentiel",'year'] = temp[liste_ocsge.year=="différentiel"][3] + ' - ' + temp[liste_ocsge.year=="différentiel"][4]
liste_ocsge = liste_ocsge[['codgeo', 'libgeo', 'year', 'link']]

In [9]:
liste_ocsge.head(5)

,codgeo,libgeo,year,link
0,01,Ain,2021,https://data.geopf.fr/telechargement/download/...
1,01,Ain,2018,https://data.geopf.fr/telechargement/download/...
2,01,Ain,2018 - 2021,https://data.geopf.fr/telechargement/download/...
3,38,Isère,2021,https://data.geopf.fr/telechargement/download/...
4,38,Isère,2018,https://data.geopf.fr/telechargement/download/...


Traitement sur des départements spécifiques

In [10]:
liste_dep = ['09', '11', '12', '30', '31', '32', '34', '46', '48', '65', '66', '81', '82']

In [11]:
liste_ocsge_dep = liste_ocsge[liste_ocsge.codgeo.isin(liste_dep)].copy()

In [22]:
url = liste_ocsge_dep.link.iloc[0]

In [16]:
r = requests.get(link, link.rsplit('/', 1)[1])

In [25]:
with requests.get(url, stream=True) as response:
    if response.status_code == 200:
        with open(link.rsplit('/', 1)[1], 'wb') as file:
            for chunk in response.iter_content(chunk_size=8192):
                file.write(chunk)

In [50]:
search_string = "OCCUPATION_SOL"
with py7zr.SevenZipFile(link.rsplit('/', 1)[1], mode='r') as archive:
        # Get the list of files in the archive
        all_files = archive.getnames()

        # Filter files that contain the search string
        filtered_files = [f for f in all_files if search_string in f]

        # Extract only the filtered files
        if filtered_files:
          archive.extract(targets=filtered_files, path='ocsge_temp',recursive=False)

        for file in filtered_files:
                source_file = os.path.join('ocsge_temp', file)
                destination_file = os.path.join('ocsge', os.path.basename(file))
                os.makedirs(os.path.dirname(destination_file), exist_ok=True)
                if os.path.exists(source_file):  # Only move if the file was extracted
                    os.rename(source_file, destination_file)

In [51]:
ocsge = gpd.read_file('ocsge/OCCUPATION_SOL.shp')

In [52]:
ocsge

,ID,CODE_CS,CODE_US,MILLESIME,SOURCE,OSSATURE,ID_ORIGINE,CODE_OR,geometry
0,OCSGE0000000010038251401,CS1.1.1.1,US5,2022,calcul,0,NC,NC,"POLYGON ((592562.88 6226166.63, 592567.52 6226..."
1,OCSGE0000000010038251464,CS1.1.1.1,US5,2022,calcul,0,NC,NC,"POLYGON ((592910.11 6238895.02, 592907.31 6238..."
2,OCSGE0000000010038251465,CS1.1.1.1,US5,2022,calcul,0,NC,NC,"POLYGON ((592885.16 6238868.6, 592869 6238869...."
3,OCSGE0000000010038251520,CS1.1.1.1,US5,2022,calcul,0,NC,NC,"POLYGON ((592329.13 6239377.69, 592327.94 6239..."
4,OCSGE0000000010038251582,CS1.1.1.1,US5,2022,calcul,0,NC,NC,"POLYGON ((592991.78 6239701.25, 592989.79 6239..."
...,...,...,...,...,...,...,...,...,...
166056,OCSGE0000000010038415326,CS2.2.1,US6.3,2022,calcul,0,NC,NC,"POLYGON ((553537.62 6200816.41, 553550.31 6200..."
166057,OCSGE0000000010038415327,CS2.2.1,US6.3,2022,calcul,0,NC,NC,"POLYGON ((572973.94 6177933.88, 572972 6177933..."
166058,OCSGE0000000010038415328,CS2.2.1,US6.3,2022,calcul,0,NC,NC,"POLYGON ((572926.2 6178056.53, 572924.62 61780..."
166059,OCSGE0000000010038415329,CS2.2.1,US6.3,2022,calcul,0,NC,NC,"POLYGON ((572842.2 6178373.6, 572843.6 6178379..."
